# HeartWise — Smart Prediction of Heart Disease Using Machine Learning in R

Educational classification project using logistic regression and random forest. This notebook is not for clinical use.

## Setup
Install the required packages (if needed), then load them.

In [ ]:
packages <- c("tidyverse", "caret", "randomForest", "ROCR", "corrplot")
missing_packages <- packages[!packages %in% rownames(installed.packages())]
if (length(missing_packages)) install.packages(missing_packages)
invisible(lapply(packages, library, character.only = TRUE))

## Load and inspect the data
Place `heart.csv` in the notebook's working directory. The dataset should include a binary `target` column.

In [ ]:
heart <- read.csv("heart.csv")
head(heart)
str(heart)
summary(heart)

## Data preprocessing
Check for missing values, remove duplicate records, and encode categorical variables.

In [ ]:
sum(is.na(heart))
heart <- heart[!duplicated(heart), ]
categorical_columns <- intersect(c("sex", "cp", "fbs", "exang", "target"), names(heart))
heart[categorical_columns] <- lapply(heart[categorical_columns], factor)
heart$target <- factor(heart$target, levels = c("0", "1"))
str(heart)

## Exploratory data analysis

In [ ]:
ggplot(heart, aes(x = target)) +
  geom_bar(fill = "tomato") +
  labs(title = "Heart Disease Outcome Distribution", x = "Target", y = "Count")

ggplot(heart, aes(x = age)) +
  geom_histogram(binwidth = 5, fill = "steelblue", color = "white") +
  labs(title = "Age Distribution", x = "Age", y = "Count")

ggplot(heart, aes(x = sex, fill = target)) +
  geom_bar(position = "dodge") +
  labs(title = "Outcome by Sex", x = "Sex", y = "Count")

### Numeric feature correlations

In [ ]:
numeric_data <- dplyr::select(heart, where(is.numeric))
cor_matrix <- cor(numeric_data, use = "pairwise.complete.obs")
corrplot::corrplot(cor_matrix, method = "color", type = "upper", tl.cex = 0.8)

## Train-test split
A fixed seed makes the split reproducible. The test set is held out for evaluation.

In [ ]:
set.seed(123)
index <- caret::createDataPartition(heart$target, p = 0.75, list = FALSE)
train <- heart[index, , drop = FALSE]
test <- heart[-index, , drop = FALSE]
nrow(train)
nrow(test)

## Logistic regression
The model estimates the probability of target class `1`. A 0.4 probability threshold is used below for the example classification.

In [ ]:
model_lr <- glm(target ~ ., data = train, family = binomial())
summary(model_lr)
prob_lr <- predict(model_lr, newdata = test, type = "response")
pred_lr <- factor(ifelse(prob_lr > 0.4, "1", "0"), levels = levels(test$target))
set.seed(10)
sample_rows <- sample(seq_len(nrow(test)), min(10, nrow(test)))
data.frame(Actual = test$target[sample_rows], Predicted = pred_lr[sample_rows])

In [ ]:
cm_lr <- caret::confusionMatrix(pred_lr, reference = test$target, positive = "1")
cm_lr
round(cm_lr$overall["Accuracy"] * 100, 2)

### Logistic regression ROC curve

In [ ]:
roc_obj <- ROCR::prediction(prob_lr, test$target == "1")
roc_perf <- ROCR::performance(roc_obj, "tpr", "fpr")
plot(roc_perf, col = "blue", main = "ROC Curve — Logistic Regression")
abline(0, 1, lty = 2, col = "gray")

## Random forest
Train a tree-based model for comparison. Class probabilities are taken from the `1` outcome level.

In [ ]:
set.seed(123)
rf_model <- randomForest::randomForest(
  target ~ ., data = train, ntree = 100, mtry = 3, maxnodes = 15, importance = TRUE
)
rf_prob <- predict(rf_model, newdata = test, type = "prob")[, "1"]
rf_pred <- factor(ifelse(rf_prob > 0.5, "1", "0"), levels = levels(test$target))
data.frame(Actual = test$target[sample_rows], Predicted = rf_pred[sample_rows])

In [ ]:
cm_rf <- caret::confusionMatrix(rf_pred, reference = test$target, positive = "1")
cm_rf
round(cm_rf$overall["Accuracy"] * 100, 2)

### Random forest feature importance

In [ ]:
randomForest::varImpPlot(rf_model)

## Export data
This optional export can be opened in other analysis tools.

In [ ]:
write.csv(heart, "heart_tableau.csv", row.names = FALSE)

## Example prediction helper
New data must contain the same predictor columns and factor levels used to train the model. Output is an illustrative model classification, not a medical assessment.

In [ ]:
predict_heart <- function(newdata, threshold = 0.5) {
  probability <- predict(model_lr, newdata = newdata, type = "response")
  ifelse(probability > threshold, "High model score", "Low model score")
}

## Disclaimer
This notebook is for educational and research purposes only. It is not intended for clinical diagnosis, treatment, or decision-making. Do not use its predictions as a substitute for advice from a qualified healthcare professional. Performance on a single train-test split does not establish real-world clinical validity.